## Bronze Layer - Class-Based Ingestion

This notebook uses a single **`BronzeIngestion`** class to load all raw files (CSV & JSON) into Bronze Delta tables.

Every entity follows the same pipeline:
1. **Define schema** — explicit StructType or DDL string (no schema inference)
2. **Read** — CSV or JSON with format-specific options (e.g. `header=True`, `multiLine=True`)
3. **Add metadata** — `ingestion_timestamp` and `source_file` for audit tracking
4. **Write** — overwrite into a Unity Catalog Delta table

The shared `_ingest()` method handles the read → enrich → write flow. Each public method (`ingest_circuits`, `ingest_races`, etc.) only provides what is unique: the schema, file path, and table name.

In [0]:
catalog_name = "formula1"
bronze_schema = "practice_class"
loding_folder_path = '/Volumes/formula1/landing/files'

In [0]:

# MAGIC ## Bronze Layer - Class-Based Ingestion
# MAGIC All 6 ingestion notebooks consolidated into a single reusable class.
# MAGIC Every entity follows the same pattern so it is better that we have one clas:
# MAGIC   1. Set source path and target table name and configure all the functions before 
# MAGIC   2. Define schema for each tables
# MAGIC   3. Read file (CSV or JSON)
# MAGIC   4. Add ingestion metadata
# MAGIC   5. Write to Delta table

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, FloatType, DateType
)
from pyspark.sql.functions import current_timestamp, col


class BronzeIngestion:
    """
    Handles ingestion of raw CSV and JSON files into Bronze Delta tables.
    All entities share the same read -> enrich -> write pipeline.
    Only the schema, source path, format, and table name differ per entity.
    """
    def __init__(self, spark: SparkSession, catalog: str, bronze_schema: str, landing_path: str):
        """
        Parameters
        ----------
        spark         : activate the active SparkSession
        catalog       : Unity Catalog name, e.g. 'formula1'
        bronze_schema : schema name inside the catalog, e.g. 'bronze'
        landing_path  : root path of the landing volume,
                        e.g. '/Volumes/formula1/landing/files' these are acquired from the bronze env steup 
        """
        self.spark = spark
        self.catalog = catalog
        self.bronze_schema = bronze_schema
        self.landing_path = landing_path

    #helper functions
    def _table(self, name: str) -> str:
        """this is the target table that we are going tosave in the bronze catalog"""
        return f"{self.catalog}.{self.bronze_schema}.{name}" # name will be given when working with the tables below

    def _source(self, relative_path: str) -> str:
        """the solurce file path basically from where we collect the data from """
        return f"{self.landing_path}/{relative_path}"

    def _add_metadata(self, df: DataFrame) -> DataFrame: # also you dont have tho specify function with schema ef g def add_metadata (self,df): that's it 
        """
        Adds two audit columns to any DataFrame:
        - ingestion_timestamp : when this load ran
        - source_file         : which file the row came from
        """
        return (
            df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", col("_metadata.file_path"))
        )

    def _write(self, df: DataFrame, table_name: str, overwrite_schema: bool = True) -> None:
        """Writes a DataFrame to a Bronze Delta table."""
        writer = (
            df
            .write
            .format("delta")
            .mode("overwrite")
        )
        #we have it in the function defined above so we have to make a condition
        if overwrite_schema:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(table_name)

    def _ingest(self, file_format: str, source_path: str, table_name: str, schema, read_options: dict = None) -> DataFrame:
        """
        Core ingestion pipeline shared by every entity:
        read -> enrich -> write -> return final DataFrame.

        Parameters
      
        file_format   : 'csv' or 'json'
        source_path   : full path to the file or folder
        table_name    : fully qualified Delta table name
        schema        : StructType or DDL string
        read_options  : extra .option(k, v) pairs as a dict, e.g. {'multiLine': True}
        """
        reader = self.spark.read.format(file_format).schema(schema)

        # CSV files need a header option; JSON files do not
        if file_format == "csv":
            reader = reader.option("header", True)

        # Apply any extra options (e.g. multiLine for sprints) for the multiline jason file 
        if read_options:
            for key, value in read_options.items():
                reader = reader.option(key, value)

        raw_df = reader.load(source_path)
        final_df = self._add_metadata(raw_df)
        self._write(final_df, table_name)
        return final_df


    # Public entity methods
   
    def ingest_circuits(self) -> DataFrame:
        """Ingests circuits.csv -> formula1.bronze.circuits"""
        schema = StructType([
            StructField("circuitId",   StringType(), True),
            StructField("url",         StringType(), True),
            StructField("circuitName", StringType(), True),
            StructField("lat",         DoubleType(),  True),
            StructField("long",        DoubleType(),  True),
            StructField("locality",    StringType(), True),
            StructField("country",     StringType(), True),
        ])
        return self._ingest(
            file_format="csv",
            source_path=self._source("circuits.csv"),
            table_name=self._table("circuits_class"),
            schema=schema
        )

    def ingest_races(self) -> DataFrame:
        """Ingests races.csv -> formula1.bronze.races"""
        schema = StructType([
            StructField("season",    IntegerType(), True),
            StructField("round",     IntegerType(), True),
            StructField("url",       StringType(),  True),
            StructField("raceName",  StringType(),  True),
            StructField("date",      DateType(),    True),
            StructField("circuitId", StringType(),  True),
        ])
        return self._ingest(
            file_format="csv",
            source_path=self._source("races.csv"),
            table_name=self._table("races_class"),
            schema=schema
        )

    def ingest_constructors(self) -> DataFrame:
        """Ingests constructors.json -> formula1.bronze.constructors"""
        schema = "constructorId STRING, name STRING, nationality STRING, url STRING"
        return self._ingest(
            file_format="json",
            source_path=self._source("constructors.json"),
            table_name=self._table("constructors_class"),
            schema=schema
        )

    def ingest_drivers(self) -> DataFrame:
        """Ingests drivers.json (nested name field) -> formula1.bronze.drivers"""
        name_schema = StructType([
            StructField("givenName",  StringType(), True),
            StructField("familyName", StringType(), True),
        ])
        schema = StructType([
            StructField("driverId",    StringType(), True),
            StructField("name",        name_schema),
            StructField("dateOfBirth", DateType(),   True),
            StructField("nationality", StringType(), True),
            StructField("URL",         StringType(), True),
        ])
        return self._ingest(
            file_format="json",
            source_path=self._source("drivers.json"),
            table_name=self._table("drivers_class"),
            schema=schema,
            read_options={"mode": "FAILFAST"}
        )

    def ingest_results(self) -> DataFrame:
        """Ingests results/ folder (multiple JSON files) -> formula1.bronze.results"""
        schema = (
            "date DATE, raceName STRING, round INT, season INT, url STRING, "
            "constructorId STRING, driverId STRING, grid INT, laps INT, "
            "number INT, points DOUBLE, position INT, positionText STRING, status STRING"
        )
        return self._ingest(
            file_format="json",
            source_path=self._source("results"),
            table_name=self._table("results_class"),
            schema=schema
        )

    def ingest_sprints(self) -> DataFrame:
        """Ingests sprints/ folder (multi-line JSON files) -> formula1.bronze.sprints"""
        schema = StructType([
            StructField("date",          StringType(),  True),
            StructField("raceName",      StringType(),  True),
            StructField("round",         IntegerType(), True),
            StructField("season",        IntegerType(), True),
            StructField("url",           StringType(),  True),
            StructField("constructorId", StringType(),  True),
            StructField("driverId",      StringType(),  True),
            StructField("grid",          IntegerType(), True),
            StructField("laps",          IntegerType(), True),
            StructField("number",        IntegerType(), True),
            StructField("points",        FloatType(),   True),
            StructField("position",      IntegerType(), True),
            StructField("positionText",  StringType(),  True),
            StructField("status",        StringType(),  True),
        ])
        return self._ingest(
            file_format="json",
            source_path=self._source("sprints"),
            table_name=self._table("sprints_class"),
            schema=schema,
            read_options={"multiLine": True}
        )

    def ingest_all(self) -> None:
        """Runs all 6 ingestion methods in sequence."""
        self.ingest_circuits()
        self.ingest_races()
        self.ingest_constructors()
        self.ingest_drivers()
        self.ingest_results()
        self.ingest_sprints()
        print("All Bronze tables loaded successfully.")


bronze = BronzeIngestion(
    spark=spark,
    catalog=catalog_name,
    bronze_schema=bronze_schema,
    landing_path=loding_folder_path  # keeping your variable name as-is
)

bronze.ingest_all()